# S16 · Return, risk, and how stocks move together

We take a few NSE stocks, measure how much they earn (their **return**) and how
much they bounce around (their **risk**), and build the table Σ that records how
each pair of them moves together. Then we mix the stocks and watch the risk drop
below the risk of the typical single stock. That drop is diversification, and we
see it with our own eyes.

**New here? Read this once.**

- New to Python? You can still do this whole notebook. Press the play button on
  each cell, top to bottom, and read the plain-English note above each one.
- New to "risk" or "moving together" as numbers? Open the primer
  `primers/diversification_and_covariance.md` for a ten-minute, picture-first
  version.
- Already confident with code or with mean-variance theory? Skip to the cell
  marked **Stretch (optional)** near the end.
- Stuck on a word? It is in `primers/glossary.md`.

## Setup

On **Google Colab**, run the next cell once. On your **own machine** you already
installed everything with `uv`, so it does nothing there.

In [ ]:
# This notebook tries to download live stock prices with yfinance, which Colab
# does not always ship. cvxpy is not needed until notebook 2.
import sys
if "google.colab" in sys.modules:
    !pip install -q yfinance
else:
    print("Not on Colab - assuming the libraries are already installed.")

## Step 1 — get our tools ready

We use NumPy for maths on lists of numbers, pandas for tables of data, and
matplotlib for charts. We also set a random seed, so that if we ever fall back to
made-up data it is the same every time you run the notebook.

In [ ]:
import numpy as np                 # fast maths on lists of numbers
import pandas as pd                # tables of data
import matplotlib.pyplot as plt    # charts

# A seed makes any random numbers reproducible - you get the same result twice.
np.random.seed(0)

print("Tools ready.")

## Step 2 — get some stock prices

We try to download three years of daily prices for five well-known Indian stocks.
If there is no internet (say, in an exam hall), the cell quietly makes up a
realistic set of prices instead, so the notebook always runs. Read the comments:
this is the same data helper we reuse in every notebook today.

In [ ]:
# We try to download REAL daily prices from Yahoo Finance.
# If there is no internet, we instead build a synthetic - made up but realistic -
# set of prices, so this notebook ALWAYS runs. We print which one happened.

# The stocks we want. The '.NS' suffix means the National Stock Exchange of India.
stock_tickers = ["TCS.NS", "INFY.NS", "RELIANCE.NS", "ITC.NS", "HDFCBANK.NS"]

used_live_data = False   # we will flip this to True if the download works

try:
    import yfinance as yf

    # Download about three years of daily prices for all the stocks at once.
    raw = yf.download(stock_tickers, period="3y", auto_adjust=True, progress=False)

    # yfinance gives us several price types (Open, High, Low, Close...).
    # We keep the 'Close' price - the price at the end of each trading day.
    prices = raw["Close"]

    # Drop any days where a price is missing, and keep the columns in our order.
    prices = prices.dropna()
    prices = prices[stock_tickers]

    # If we somehow got an empty table, treat it as a failure.
    if len(prices) < 50:
        raise ValueError("not enough price data came back")

    used_live_data = True
    print("Downloaded LIVE price data from Yahoo Finance.")

except Exception as problem:
    # ---- Offline fallback: build synthetic prices with NumPy. -------------
    print("Could not download live data, so we are using SYNTHETIC data instead.")
    print("(Reason:", problem, ")")

    number_of_stocks = len(stock_tickers)
    number_of_days = 750

    # Each stock has its own average daily return and its own wobble (risk).
    # We pick believable yearly numbers and convert them to daily numbers.
    yearly_return = np.random.uniform(0.06, 0.20, size=number_of_stocks)
    yearly_risk = np.random.uniform(0.18, 0.40, size=number_of_stocks)
    daily_average_return = yearly_return / 252.0      # ~252 trading days a year
    daily_risk = yearly_risk / np.sqrt(252.0)

    # To make the stocks move TOGETHER a little (as real stocks do), we mix a few
    # shared 'market mood' signals into each stock's daily return.
    number_of_shared_factors = 3
    loadings = np.random.uniform(0.2, 1.0, size=(number_of_stocks, number_of_shared_factors))
    shared_moods = np.random.normal(0, 1, size=(number_of_days, number_of_shared_factors))
    own_wiggle = np.random.normal(0, 1, size=(number_of_days, number_of_stocks))

    # Combine shared moods and each stock's own wiggle into daily returns.
    daily_returns_matrix = shared_moods @ loadings.T + own_wiggle

    # Scale each column so it has the daily average return and risk we chose.
    for stock_number in range(number_of_stocks):
        column = daily_returns_matrix[:, stock_number]
        column = (column - column.mean()) / column.std()      # standardise
        column = column * daily_risk[stock_number] + daily_average_return[stock_number]
        daily_returns_matrix[:, stock_number] = column

    # Turn daily returns into a price series starting at 100 for every stock.
    price_paths = 100 * np.cumprod(1 + daily_returns_matrix, axis=0)
    business_days = pd.bdate_range(end=pd.Timestamp.today(), periods=number_of_days)
    prices = pd.DataFrame(price_paths, index=business_days, columns=stock_tickers)

# Show the first few rows so we can see what we are working with.
print("\nPrice table shape (rows = days, columns = stocks):", prices.shape)
prices.head()

## Step 3 — always look at the data first

Each line is one stock's price over three years. They start at different levels, so
they are hard to compare directly. That is why we will switch to *returns* next,
which put every stock on the same scale.

In [ ]:
plt.figure(figsize=(9, 5))

# Draw one line per stock.
for one_stock in prices.columns:
    plt.plot(prices.index, prices[one_stock], label=one_stock)

plt.xlabel("date")
plt.ylabel("price")
plt.title("Daily prices of our stocks")
plt.legend()
plt.show()

## Step 4 — turn prices into daily returns

A **return** is the percentage change in price from one day to the next. Returns
matter more than the raw price because they put a ₹50 stock and a ₹5000 stock on
the same footing. `pct_change()` does exactly this; the first day has no day before
it, so we drop it.

In [ ]:
# Daily return = (today's price - yesterday's price) / yesterday's price.
daily_returns = prices.pct_change().dropna()

print("Daily returns table shape:", daily_returns.shape)
daily_returns.head()

## Step 5 — the average return of each stock

We average each stock's daily returns, then scale up to a yearly figure (about 252
trading days in a year). This gives μ (the Greek letter 'mu'): the return we expect
from each stock in a year. Higher is better.

In [ ]:
trading_days_per_year = 252

# Average daily return for each stock, then annualise it.
average_daily_return = daily_returns.mean()
mu = average_daily_return * trading_days_per_year      # mu = expected yearly return

print("Expected yearly return of each stock:")
print((mu * 100).round(2))    # shown as a percentage

## Step 6 — the risk of each stock

Risk is how much the returns bounce around. We measure it with the **standard
deviation** of the daily returns, which is just the typical distance of the returns
from their average. In finance this is called **volatility**. Again we scale it to a
year. A bigger number means a bumpier, riskier ride.

In [ ]:
# Standard deviation of daily returns = how much they wobble.
# We annualise volatility by multiplying by the square root of 252.
daily_volatility = daily_returns.std()
yearly_volatility = daily_volatility * np.sqrt(trading_days_per_year)

print("Yearly risk (volatility) of each stock:")
print((yearly_volatility * 100).round(2))    # shown as a percentage

## Step 7 — the covariance matrix Σ

Risk is not just about each stock alone. It is about how stocks move *together*.
The **covariance matrix** Σ records this for every pair of stocks at once. Each
number is a togetherness score for a pair: positive if they tend to rise and fall
together, near zero if there is no pattern. The diagonal (a stock paired with
itself) is just that stock's own spread. We compute it from the daily returns and
annualise it.

In [ ]:
# .cov() gives the covariance matrix of the daily returns.
# We annualise it by multiplying by 252 (it scales with time, like variance).
Sigma = daily_returns.cov() * trading_days_per_year

print("The covariance matrix Sigma (one row and column per stock):")
Sigma.round(4)

## Step 8 — see Σ as a heatmap

A table of numbers is hard to read, so we colour it. Darker squares mean the two
stocks move together more strongly. The diagonal (top-left to bottom-right) is
darkest because every stock moves perfectly with itself.

In [ ]:
stock_names = list(prices.columns)

plt.figure(figsize=(6.5, 5.5))
plt.imshow(Sigma.values, cmap="Blues")
plt.colorbar(label="covariance")

# Put the stock names on both axes.
plt.xticks(range(len(stock_names)), stock_names, rotation=45, ha="right")
plt.yticks(range(len(stock_names)), stock_names)
plt.title("Covariance matrix Sigma (darker = move together more)")
plt.tight_layout()
plt.show()

## Step 9 — the return and risk of a whole mix

A **portfolio** is a mix: a list of **weights** `w` saying what fraction of our
money goes into each stock. The weights add up to 1.

- The portfolio's expected return is the simple weighted average of the stock
  returns: put more weight on a stock and its return counts for more.
- The portfolio's risk is **not** a simple average. It depends on how the stocks
  move together, which is exactly what Σ holds. The one line
  `weights @ Sigma @ weights` combines all the togetherness scores into the mix's
  variance for us.

Let us compute both for an equal-weight portfolio (the same amount in each
stock).

In [ ]:
number_of_stocks = len(stock_names)

# Equal weights: put the same fraction of money in every stock.
equal_weights = np.ones(number_of_stocks) / number_of_stocks
print("Weights (they sum to 1):", equal_weights.round(3), " sum =", equal_weights.sum())

# Portfolio expected return = weighted average of the stocks' returns.
portfolio_return = equal_weights @ mu.values

# Portfolio variance combines every pair's covariance via Sigma.
portfolio_variance = equal_weights @ Sigma.values @ equal_weights

# Risk (volatility) is the square root of the variance.
portfolio_risk = np.sqrt(portfolio_variance)

print("\nEqual-weight portfolio expected yearly return:", round(portfolio_return * 100, 2), "%")
print("Equal-weight portfolio yearly risk            :", round(portfolio_risk * 100, 2), "%")

## Step 10 — watch diversification reduce risk

Here is the magic from the primer, in numbers. If we spread our money across
stocks that do **not** move in lockstep, the bumps partly cancel and the mix is
calmer than the stocks are on their own. To show it, we compare two numbers:

- the plain average of the individual stock risks (what you might naively expect),
- the actual risk of the equal-weight portfolio (what really happens).

The portfolio risk is lower. That gap is diversification at work.

In [ ]:
# The plain average of each stock's own risk.
average_individual_risk = yearly_volatility.mean()

print("Average of the individual stock risks:", round(average_individual_risk * 100, 2), "%")
print("Actual equal-weight portfolio risk   :", round(portfolio_risk * 100, 2), "%")

risk_saved = average_individual_risk - portfolio_risk
print("\nRisk removed by diversifying         :", round(risk_saved * 100, 2), "percentage points")
print("The mix is calmer than the typical single stock - that is diversification.")

### Stretch (optional) — build the portfolio risk by hand

Skip this if you are new to code. If you want to see what
`weights @ Sigma @ weights` actually does, here it is spelled out. The portfolio
variance is the sum, over every pair of stocks (i, j), of
`weight_i * weight_j * covariance(i, j)`. We add all those pieces up with two plain
loops and check we get the same number as the one-liner above.

In [ ]:
Sigma_values = Sigma.values

# Add up weight_i * weight_j * covariance(i, j) over every pair of stocks.
variance_by_hand = 0.0
for i in range(number_of_stocks):
    for j in range(number_of_stocks):
        variance_by_hand = variance_by_hand + equal_weights[i] * equal_weights[j] * Sigma_values[i, j]

print("Portfolio variance, one-liner :", round(portfolio_variance, 6))
print("Portfolio variance, by hand   :", round(variance_by_hand, 6))
print("\nSame number - the one-liner is just this double sum done fast.")

## What you just did

You measured the return and risk of several NSE stocks, built the covariance table
Σ that records how they move together, and saw that mixing them gives **less** risk
than the stocks have on their own. The mix's risk, `weights @ Sigma @ weights`, is
the quantity we will now *minimise* with CVXPY to find the best possible
portfolios. On to notebook 2.